In [ ]:
import os
import glob
import shutil
import pickle

# ============================================================
# 1. Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


# ============================================================
# 2. Install kagglehub
# ============================================================

!pip install --quiet kagglehub

import kagglehub


# ============================================================
# 3. Download the KITTI dataset from Kaggle
# ============================================================

print("Downloading from Kaggle...")

dataset_path = kagglehub.dataset_download(
    "garymk/kitti-3d-object-detection-dataset"
)

print(f"Dataset downloaded in:")
print(dataset_path)


# ============================================================
# 4. Define the structure in Google Drive
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/3D_Perception/data/kitti_validation"

IMAGE_DIR = os.path.join(DRIVE_ROOT, "image_2")
VELO_DIR = os.path.join(DRIVE_ROOT, "velodyne_reduced")
CALIB_DIR = os.path.join(DRIVE_ROOT, "calib")
LABEL_DIR = os.path.join(DRIVE_ROOT, "label_2")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(VELO_DIR, exist_ok=True)
os.makedirs(CALIB_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)


# ============================================================
# 5. Find kitti_infos_val.pkl
# ============================================================

pkl_files = glob.glob(
    f"{dataset_path}/**/kitti_infos_val.pkl",
    recursive=True
)

if not pkl_files:
    raise FileNotFoundError(
        " kitti_infos_val.pkl didn't found in the dataset."
    )

val_pkl = pkl_files[0]

print(f"\nkitti_infos_val.pkl found:")
print(val_pkl)


# ============================================================
# 6. Create the validation set
# ============================================================

with open(val_pkl, "rb") as f:
    val_infos = pickle.load(f)

print(f"\nNumber of validation frames: {len(val_infos)}")


# ============================================================
# 7. Copy only the validation set frames
# ============================================================

for i, info in enumerate(val_infos):

    # --------------------------------------------------------
    # Retrive the LiDAR path from the .pkl file
    # --------------------------------------------------------

    velo_rel = info["velodyne_path"]

    # Example:
    # training/velodyne/000123.bin

    frame_id = os.path.splitext(
        os.path.basename(velo_rel)
    )[0]

    # --------------------------------------------------------
    # velodyne_reduced path
    # --------------------------------------------------------

    reduced_velo_rel = velo_rel.replace(
        "training/velodyne/",
        "training/velodyne_reduced/"
    )

    reduced_velo_path = os.path.join(
        dataset_path,
        reduced_velo_rel
    )

    # --------------------------------------------------------
    # Others KITTI files
    # --------------------------------------------------------

    image_path = os.path.join(
        dataset_path,
        "training",
        "image_2",
        f"{frame_id}.png"
    )

    calib_path = os.path.join(
        dataset_path,
        "training",
        "calib",
        f"{frame_id}.txt"
    )

    label_path = os.path.join(
        dataset_path,
        "training",
        "label_2",
        f"{frame_id}.txt"
    )

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    required_files = {
        "velodyne_reduced": reduced_velo_path,
        "image": image_path,
        "calib": calib_path,
        "label": label_path
    }

    for name, file_path in required_files.items():

        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"\n {name} didn't found:\n{file_path}"
            )

    # --------------------------------------------------------
    # Copy in Google Drive
    # --------------------------------------------------------

    shutil.copy2(
        reduced_velo_path,
        os.path.join(VELO_DIR, f"{frame_id}.bin")
    )

    shutil.copy2(
        image_path,
        os.path.join(IMAGE_DIR, f"{frame_id}.png")
    )

    shutil.copy2(
        calib_path,
        os.path.join(CALIB_DIR, f"{frame_id}.txt")
    )

    shutil.copy2(
        label_path,
        os.path.join(LABEL_DIR, f"{frame_id}.txt")
    )

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 100 == 0:
        print(
            f"Copied {i + 1}/{len(val_infos)} frames..."
        )


# ============================================================
# 8. Final check
# ============================================================

print("\n==============================================")
print("✓ VALIDATION SET COPIED IN GOOGLE DRIVE")
print("==============================================")

print(f"Total frames: {len(val_infos)}")

print(f"\nImages:        {len(os.listdir(IMAGE_DIR))}")
print(f"LiDAR reduced:   {len(os.listdir(VELO_DIR))}")
print(f"Calibrations:    {len(os.listdir(CALIB_DIR))}")
print(f"Labels:          {len(os.listdir(LABEL_DIR))}")

print("\nPath:")
print(DRIVE_ROOT)

